In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from google.cloud import bigquery

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [3]:
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT_DIR / ".env")

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
MARTS_DATASET = os.getenv("BQ_MARTS_DATASET", "growthpilot_marts")
KEY_PATH = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

if KEY_PATH and KEY_PATH.startswith("./"):
    KEY_PATH = str((ROOT_DIR / KEY_PATH).resolve())

client = bigquery.Client.from_service_account_json(KEY_PATH, project=PROJECT_ID)

PROJECT_ID, MARTS_DATASET

('growthpilot-ai-496111', 'growthpilot_marts')

In [5]:
query = f"""
select
    user_id,
    traffic_source,
    country,
    city,
    age,
    gender,
    first_purchase_date,
    last_purchase_date,
    recency_days,
    frequency,
    total_items_purchased,
    monetary_value,
    avg_order_value
from `{PROJECT_ID}.{MARTS_DATASET}.mart_customer_rfm`
where user_id is not null
"""

df = client.query(query).to_dataframe()
df.head()

/Users/hao/Desktop/personal_repositories/growthpilot-ai/.venv/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,user_id,traffic_source,country,city,age,gender,first_purchase_date,last_purchase_date,recency_days,frequency,total_items_purchased,monetary_value,avg_order_value
0,32952,Search,China,Shenzhen,66,M,2019-02-04,2019-02-04,2659,1,1,282.00,282.00
1,87725,Search,Brasil,Mucuri,18,M,2019-02-08,2019-02-08,2655,1,1,178.00,178.00
2,23096,Search,United States,Seattle,58,M,2019-02-12,2019-02-12,2651,1,1,26.95,26.95
3,58221,Organic,Australia,Toolara Forest,63,M,2019-02-14,2019-02-14,2649,1,1,79.50,79.50
4,5039,Email,Brasil,Barcarena,70,M,2019-02-20,2019-02-20,2643,1,1,26.00,26.00


In [6]:
df[
    [
        "recency_days",
        "frequency",
        "total_items_purchased",
        "monetary_value",
        "avg_order_value",
    ]
].describe()

,recency_days,frequency,total_items_purchased,monetary_value,avg_order_value
count,66227.0,66227.0,66227.0,66227.000000,66227.000000
mean,581.873692,1.413985,2.055853,122.365627,86.627162
std,559.689564,0.716396,1.415838,127.033909,85.778025
min,0.0,1.0,1.0,0.020000,0.020000
25%,132.0,1.0,1.0,38.950000,33.990000
50%,401.0,1.0,2.0,79.990000,60.320000
75%,885.0,2.0,3.0,162.990000,110.000000
max,2672.0,4.0,12.0,2045.500000,1247.940000


In [7]:
df[["frequency", "monetary_value", "avg_order_value"]].quantile(
    [0.5, 0.75, 0.9, 0.95, 0.99]
)

,frequency,monetary_value,avg_order_value
0.50,1.0,79.9900,60.3200
0.75,2.0,162.9900,110.0000
0.90,2.0,275.1340,179.9840
0.95,3.0,364.0000,239.4900
0.99,4.0,596.8992,405.4796


In [8]:
seg_df = df.copy()

seg_df["recency_days"] = seg_df["recency_days"].fillna(seg_df["recency_days"].median())
seg_df["frequency"] = seg_df["frequency"].fillna(0)
seg_df["monetary_value"] = seg_df["monetary_value"].fillna(0)
seg_df["avg_order_value"] = seg_df["avg_order_value"].fillna(0)

seg_df["frequency_log"] = np.log1p(seg_df["frequency"])
seg_df["monetary_log"] = np.log1p(seg_df["monetary_value"])

features = seg_df[["recency_days", "frequency_log", "monetary_log"]]

scaler = StandardScaler()
X = scaler.fit_transform(features)

features.head()

,recency_days,frequency_log,monetary_log
0,2659,0.693147,5.645447
1,2655,0.693147,5.187386
2,2651,0.693147,3.330417
3,2649,0.693147,4.388257
4,2643,0.693147,3.295837


In [9]:
kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10,
)

seg_df["cluster_id"] = kmeans.fit_predict(X)

seg_df[
    [
        "user_id",
        "recency_days",
        "frequency",
        "monetary_value",
        "avg_order_value",
        "cluster_id",
    ]
].head()

,user_id,recency_days,frequency,monetary_value,avg_order_value,cluster_id
0,32952,2659,1,282.00,282.00,0
1,87725,2655,1,178.00,178.00,0
2,23096,2651,1,26.95,26.95,0
3,58221,2649,1,79.50,79.50,0
4,5039,2643,1,26.00,26.00,0


In [10]:
cluster_summary = (
    seg_df.groupby("cluster_id")
    .agg(
        customers=("user_id", "count"),
        avg_recency_days=("recency_days", "mean"),
        avg_frequency=("frequency", "mean"),
        avg_monetary_value=("monetary_value", "mean"),
        avg_order_value=("avg_order_value", "mean"),
        total_monetary_value=("monetary_value", "sum"),
    )
    .reset_index()
    .sort_values("total_monetary_value", ascending=False)
)

cluster_summary

,cluster_id,customers,avg_recency_days,avg_frequency,avg_monetary_value,avg_order_value,total_monetary_value
4,4,16943,369.471463,1.0,149.820506,149.820506,2538408.83
3,3,13324,376.368433,2.006455,175.245692,87.560427,2334973.60
1,1,5696,296.339185,3.275281,289.546088,89.705446,1649254.52
0,0,12111,1535.656098,1.073982,85.385738,80.555905,1034106.67
2,2,18153,384.222002,1.008318,30.141836,30.045630,547164.74


In [11]:
summary = cluster_summary.copy()

summary["recency_rank"] = summary["avg_recency_days"].rank(ascending=True)
summary["frequency_rank"] = summary["avg_frequency"].rank(ascending=False)
summary["monetary_rank"] = summary["avg_monetary_value"].rank(ascending=False)

summary["segment_score"] = (
    summary["recency_rank"] * 0.25
    + summary["frequency_rank"] * 0.35
    + summary["monetary_rank"] * 0.40
)

summary.sort_values("segment_score")

,cluster_id,customers,avg_recency_days,avg_frequency,avg_monetary_value,avg_order_value,total_monetary_value,recency_rank,frequency_rank,monetary_rank,segment_score
1,1,5696,296.339185,3.275281,289.546088,89.705446,1649254.52,1.0,1.0,1.0,1.00
3,3,13324,376.368433,2.006455,175.245692,87.560427,2334973.60,3.0,2.0,2.0,2.25
4,4,16943,369.471463,1.0,149.820506,149.820506,2538408.83,2.0,5.0,3.0,3.45
0,0,12111,1535.656098,1.073982,85.385738,80.555905,1034106.67,5.0,3.0,4.0,3.90
2,2,18153,384.222002,1.008318,30.141836,30.045630,547164.74,4.0,4.0,5.0,4.40


In [12]:
ordered_clusters = summary.sort_values("segment_score")["cluster_id"].tolist()

segment_names = [
    "High-value loyal customers",
    "Recent active buyers",
    "Potential loyal customers",
    "One-time buyers",
    "Dormant low-value customers",
]

cluster_to_segment = {
    cluster_id: segment_names[i]
    for i, cluster_id in enumerate(ordered_clusters)
}

cluster_to_segment

{1: 'High-value loyal customers',
 3: 'Recent active buyers',
 4: 'Potential loyal customers',
 0: 'One-time buyers',
 2: 'Dormant low-value customers'}

In [13]:
seg_df["segment_name"] = seg_df["cluster_id"].map(cluster_to_segment)

segment_summary = (
    seg_df.groupby("segment_name")
    .agg(
        customers=("user_id", "count"),
        avg_recency_days=("recency_days", "mean"),
        avg_frequency=("frequency", "mean"),
        avg_monetary_value=("monetary_value", "mean"),
        avg_order_value=("avg_order_value", "mean"),
        total_monetary_value=("monetary_value", "sum"),
    )
    .reset_index()
    .sort_values("total_monetary_value", ascending=False)
)

segment_summary

,segment_name,customers,avg_recency_days,avg_frequency,avg_monetary_value,avg_order_value,total_monetary_value
3,Potential loyal customers,16943,369.471463,1.0,149.820506,149.820506,2538408.83
4,Recent active buyers,13324,376.368433,2.006455,175.245692,87.560427,2334973.60
1,High-value loyal customers,5696,296.339185,3.275281,289.546088,89.705446,1649254.52
2,One-time buyers,12111,1535.656098,1.073982,85.385738,80.555905,1034106.67
0,Dormant low-value customers,18153,384.222002,1.008318,30.141836,30.045630,547164.74


In [14]:
seg_df["segmentation_method"] = "KMeans_RFM"
seg_df["segmentation_created_at"] = pd.Timestamp.utcnow()

output = seg_df[
    [
        "user_id",
        "traffic_source",
        "country",
        "city",
        "age",
        "gender",
        "first_purchase_date",
        "last_purchase_date",
        "recency_days",
        "frequency",
        "total_items_purchased",
        "monetary_value",
        "avg_order_value",
        "cluster_id",
        "segment_name",
        "segmentation_method",
        "segmentation_created_at",
    ]
].copy()

output.head()

,user_id,traffic_source,country,city,age,gender,first_purchase_date,last_purchase_date,recency_days,frequency,total_items_purchased,monetary_value,avg_order_value,cluster_id,segment_name,segmentation_method,segmentation_created_at
0,32952,Search,China,Shenzhen,66,M,2019-02-04,2019-02-04,2659,1,1,282.00,282.00,0,One-time buyers,KMeans_RFM,2026-05-13 10:08:59.069727+00:00
1,87725,Search,Brasil,Mucuri,18,M,2019-02-08,2019-02-08,2655,1,1,178.00,178.00,0,One-time buyers,KMeans_RFM,2026-05-13 10:08:59.069727+00:00
2,23096,Search,United States,Seattle,58,M,2019-02-12,2019-02-12,2651,1,1,26.95,26.95,0,One-time buyers,KMeans_RFM,2026-05-13 10:08:59.069727+00:00
3,58221,Organic,Australia,Toolara Forest,63,M,2019-02-14,2019-02-14,2649,1,1,79.50,79.50,0,One-time buyers,KMeans_RFM,2026-05-13 10:08:59.069727+00:00
4,5039,Email,Brasil,Barcarena,70,M,2019-02-20,2019-02-20,2643,1,1,26.00,26.00,0,One-time buyers,KMeans_RFM,2026-05-13 10:08:59.069727+00:00


In [15]:
ANALYTICS_DATASET = os.getenv("BQ_ANALYTICS_DATASET", "growthpilot_analytics")

In [16]:
table_id = f"{PROJECT_ID}.{ANALYTICS_DATASET}.customer_segments"

job_config = bigquery.LoadJobConfig(

    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,

    autodetect=True,

)

job = client.load_table_from_dataframe(

    output,

    table_id,

    job_config=job_config,

)

job.result()

print(f"Wrote {len(output):,} rows to {table_id}")

Wrote 66,227 rows to growthpilot-ai-496111.growthpilot_analytics.customer_segments
